# Module 2: Customer Segmentation using RFM Analysis & K-Means Clustering
### Senior Data Scientist & ML Engineering Portfolio
---
### Objectives:
1. **RFM Feature Engineering**: Extract Recency (R), Frequency (F), and Monetary (M) metrics from 396k+ verified customer transactions.
2. **Mathematical Normalization**: Address extreme right-skewness using `np.log1p()` and standardize feature scales with `StandardScaler`.
3. **Hyperparameter Selection**: Apply the **Elbow Method** (WCSS Inertia) and **Silhouette Analysis** across $K \in [2, 8]$.
4. **Behavioral Personas & Business Playbook**: Segment the 4,334 customer cohort into actionable marketing segments (*Champions*, *Loyal*, *At-Risk*, *Occasional*).


In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Calculating Behavioral RFM Features
We utilize `src.features.rfm.RFMFeatureEngineer` to compute the raw features relative to the reference date:
- **Recency**: Days elapsed since the customer's last purchase.
- **Frequency**: Total count of unique orders placed.
- **Monetary**: Cumulative GBP (£) revenue generated by the customer.


In [ ]:
from src.features.rfm import RFMFeatureEngineer

fe = RFMFeatureEngineer(data_path=Path('../data/processed/online_retail_customer_centric.csv'))
rfm_df = fe.compute_rfm()
rfm_df.head()


## 2. Statistical Skewness & Mathematical Justification for Log Transformation
Customer spend and order volume follow extreme power-law distributions.
Notice the massive gap between the 50th percentile (Median) and the Maximum values.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(rfm_df['Recency'], kde=True, ax=axes[0], color='teal').set_title('Recency (Days)')
sns.histplot(rfm_df['Frequency'], kde=True, ax=axes[1], color='coral').set_title('Frequency (Orders - Heavily Skewed)')
sns.histplot(rfm_df['Monetary'], kde=True, ax=axes[2], color='purple').set_title('Monetary (£ Spend - Power Law)')
plt.tight_layout()
plt.show()

print("Skewness metrics:")
print(rfm_df[['Recency', 'Frequency', 'Monetary']].skew().round(2))


## 3. Selecting Optimal K (Elbow Method & Silhouette Scores)
We test $K \in [2, 8]$ using `src.models.segmentation.CustomerSegmentationModel`.
- **Inertia (Within-Cluster Sum of Squares - WCSS)**: Measures cluster compactness.
- **Silhouette Score**: Evaluates separation distance between clusters ($-1$ to $+1$).


In [ ]:
from src.models.segmentation import CustomerSegmentationModel

seg_model = CustomerSegmentationModel(n_clusters=4)
seg_model.prepare_features(rfm_df)
eval_df = seg_model.evaluate_cluster_range(2, 8)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(eval_df['K'], eval_df['Inertia_WCSS'], marker='o', color='#2b5c8f', linewidth=2.5, label='Inertia (WCSS)')
ax2.plot(eval_df['K'], eval_df['Silhouette_Score'], marker='s', color='#d95f02', linewidth=2.5, linestyle='--', label='Silhouette Score')

ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', color='#2b5c8f', fontsize=12)
ax2.set_ylabel('Silhouette Score', color='#d95f02', fontsize=12)
plt.title('Optimal Cluster Selection: Elbow Curve & Silhouette Metric', fontsize=14, weight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

eval_df


## 4. Fitting Final K-Means ($K=4$) & Persona Mapping
We fit the final model and serialize the pipeline to disk.


In [ ]:
segmented_df = seg_model.fit_predict()
playbook = seg_model.get_segment_playbook()
playbook[['Segment_Name', 'CustomerCount', 'AvgRecency_Days', 'AvgFrequency_Orders', 'AvgMonetary_Spend', 'Revenue_Contribution (%)']]


## 5. Segment Revenue Contribution (Pareto Principle in Action)
Champions constitute a minority of the customer base but drive the lion's share of revenue.


In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(
    playbook['Revenue_Contribution (%)'], 
    labels=playbook['Segment_Name'], 
    autopct='%1.1f%%', 
    explode=[0.05, 0.08, 0.05, 0.05],
    colors=sns.color_palette('Set2'),
    startangle=140
)
plt.title('Gross Revenue Share by Customer Segment', fontsize=14, weight='bold')
plt.show()


## 6. Interactive 3D Cluster Visualization
Inspecting how clusters separate cleanly across Recency, Frequency, and Monetary dimensions.


In [ ]:
fig = px.scatter_3d(
    segmented_df, 
    x='Recency', 
    y='Frequency', 
    z='Monetary',
    color='Segment_Name',
    hover_name='CustomerID',
    log_y=True,
    log_z=True,
    opacity=0.7,
    title="3D Interactive Customer Segmentation Space (Log-Scale)"
)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig.show()


## 7. Executive Strategy & Retention Playbook
Concrete marketing actions tailored for each behavioral group:


In [ ]:
for _, row in playbook.iterrows():
    print(f"==================================================")
    print(f"PERSONA: {row['Segment_Name'].upper()}")
    print(f"  • Size: {row['CustomerCount']:,} customers ({row['Revenue_Contribution (%)']}% of total revenue)")
    print(f"  • Typical Profile: Recency={row['AvgRecency_Days']:.0f} days | Orders={row['AvgFrequency_Orders']:.1f} | Avg Spend=£{row['AvgMonetary_Spend']:.2f}")
    print(f"  • ACTION PLAYBOOK: {row['Recommended_Strategy']}")
